In [2]:
import numpy as np
import matplotlib.pyplot as plt
import math
import cmath

In [3]:
def parallel_resistance(r1, r2):
    return 1/((1/r1)+(1/r2))

In [6]:
# System Parameters

accumulator_series_elements = 10*14
cell_max_voltage = 4.2 #volts
cell_min_voltage = 2.5 #volts

min_pack_voltage = cell_min_voltage*accumulator_series_elements
max_pack_voltage = cell_max_voltage*accumulator_series_elements
mid_pack_voltage = (min_pack_voltage + max_pack_voltage)/2

pmax = 850 #Watts
vout = 24 #volts
iout = pmax/vout

i_in_min = pmax/max_pack_voltage
i_in_max = pmax/min_pack_voltage

acceptable_dvout = 0.5 #volts

fsw_post = 50*10**3 #Hz post rectification
fsw_pre = fsw_post/2 #Hz pre rectification aka each phase


In [7]:
#Core Parameters
core_a_l = 175*10**-9 #Henrys
saturation_flux_density = 10500 #Gauss
core_cross_section = 262 *10**-6 #m2

In [8]:
# Transformer Calculations

calc_transformer_turns_ratio = min_pack_voltage/vout
# print(calc_transformer_turns_ratio)
actual_transformer_turns_ratio = 13

min_duty_cycle = vout/(max_pack_voltage/actual_transformer_turns_ratio)
mid_duty_cycle = vout/(mid_pack_voltage/actual_transformer_turns_ratio)
max_duty_cycle = vout/(min_pack_voltage/actual_transformer_turns_ratio)

# print(max_duty_cycle)
# print(mid_duty_cycle)
# print(min_duty_cycle)

minimum_primary_turns = (max_pack_voltage*10**8)/(4.44*10**4*fsw_pre*core_cross_section*saturation_flux_density)
# print(minimum_primary_turns)

actual_primary_turns = 26
actual_secondary_turns = 2 #per secondary coil

maximum_flux_density = (max_pack_voltage*10**8)/(4.44*10**4*fsw_pre*core_cross_section*actual_primary_turns)
# print(maximum_flux_density)
flux_density_safety_factor = saturation_flux_density/maximum_flux_density
# print(flux_density_safety_factor)

primary_self_inductance = core_a_l*actual_primary_turns**2
print(primary_self_inductance*10**6) #uH


118.30000000000001


In [9]:
# FET Calculations
# https://www.ti.com/lit/an/slyt664/slyt664.pdf?ts=1745717291353&ref_url=https%253A%252F%252Fwww.google.com%252F

ton_top = (max_duty_cycle/2)/fsw_pre
print(ton_top)
ton_bot = (1-(max_duty_cycle/2))/fsw_pre
print(ton_bot)

switching_fet_rdson = 0.025 #Ohms #TODO placeholder value
switching_fet_qg = 40*10**-9 #C #TODO placeholder value
tr = 13.5*10**-9
tf = 16*10**-9

switching_fet_conduction_losses_top = ton_top*switching_fet_rdson*i_in_max**2
switching_fet_conduction_losses_bot = ton_bot*switching_fet_rdson*i_in_max**2
#TODO consider affect of ripple in current

gate_driver_source_current = 1 #A
#TODO consider asymmetrical drive currents

switching_fet_switching_losses = max_pack_voltage*i_in_max*fsw_pre*(switching_fet_qg/gate_driver_source_current)

switching_fet_total_losses_top = switching_fet_conduction_losses_top + switching_fet_switching_losses
switching_fet_total_losses_bot = switching_fet_conduction_losses_bot + switching_fet_switching_losses

print(switching_fet_total_losses_top) #Watts
print(switching_fet_total_losses_bot) #Watts

switching_fraction = (tr)*fsw_pre
print(switching_fraction)
print(1/fsw_pre)

gate_drive_source_current = 4
avg_gate_drive_current = 4*switching_fraction
print(avg_gate_drive_current)

1.782857142857143e-05
2.217142857142857e-05
1.4280026288046646
1.4280032691545188
0.0003375
4e-05
0.00135


In [11]:
# Gate Driver Calculations
bootstrap_charge_voltage = 15
bootstrap_diode_vf = 1.3
bootstrap_resistance = 2.2


I_boot_peak = (bootstrap_charge_voltage-bootstrap_diode_vf)/bootstrap_resistance
print(I_boot_peak)

switching_fet_qg = 40*10**-9 #C #TODO placeholder value
gate_driver_quiescent_current = 0.0025 #A
acceptable_chargePump_ripple = 0.5 #V

q_total_chargePump = switching_fet_qg + gate_driver_quiescent_current/fsw_pre
c_boot_min = q_total_chargePump/acceptable_chargePump_ripple
print(c_boot_min*10**9) #nF


6.227272727272727
279.99999999999994


In [8]:
# Rectifier Calculations

rectifier_fet_rdson = 0.01 #Ohms #TODO Placeholder value

In [9]:
# Equivalent Capacitance Function Definition
# https://www.power-mag.com/pdf/feature_pdf/1387888355_Murata_Feature_Layout_1.pdf

#Returned datastructure
class EquivalentCapacitance:
    def __init__(self, capacitances, fsw, ripple_current, equivalent_capacitance, equivalent_esr, ripple_voltage):
        self.capacitances = capacitances # Will be of the form [qty, capacitance (F), esr (Ohm), ripple current (A)] where the 4th column will only be added if a total ripple current was provided
        self.fsw = fsw
        self.ripple_current = ripple_current
        self.equivalent_capacitance = equivalent_capacitance
        self.equivalent_esr = equivalent_esr
        self.ripple_voltage = ripple_voltage

    def print(self):
        print(f"Capacitance Matrix: {self.capacitances}")
        print(f"Switching Frequency: {self.fsw} Hz")
        print(f"Ripple Current: {self.ripple_current} A")
        print(f"Equivalent Capacitance: {self.equivalent_capacitance} F")
        print(f"Equivalent ESR: {self.equivalent_esr} Ohms")
        print(f"Ripple Voltage: {self.ripple_voltage} V")

def equiv_capacitance(capacitances, fsw, current = 0):
    # capacitances is a 3 column, n row matrix with rows of the form [qty, capacitance (uF), esr (mOhm)]
    # fsw is the fundamental switching frequency (kHz)
    # current is optional. If provided, the return will include the ripple voltage and current per capacitor
    
    cs = np.array(capacitances)
    cs = cs.astype(float)
    xs = 1/(2*3.1416*fsw*cs[:, 1])
    
    cp = cs.copy()
    cp[:, 1] = cs[:, 1]/(1 + (cs[:, 2]/xs)**2)
    cp[:, 2] = (cs[:, 2]**2 + xs**2)/cs[:, 2]
    
    cpe = np.sum(cp[:, 0]*cp[:, 1])
    rpe = 1/np.sum(cp[:, 0]/cp[:, 2])

    xpe = 1/(2*3.1416*fsw*cpe)
    cse = cpe*(1 + (xpe/rpe)**2)
    rse = rpe/(1 + (rpe/xpe)**2)
    xse = 1/(2*3.1416*fsw*cse)

    dv = np.nan
    if(current != 0):
        dv = current*math.sqrt(rse**2 + xse**2)
        di = dv/np.sqrt(cs[:, 2]**2 + xs**2)
        cs = np.hstack((cs, di.reshape(-1, 1)))
    return EquivalentCapacitance(cs, fsw, current if current > 0 else np.nan, cse, rse, dv)


In [10]:
# Filter Calculations
# https://www.ti.com/lit/an/slva477b/slva477b.pdf?ts=1745747396768

acceptable_inductor_ripple_current = 0.4*iout #Amps
calculated_inductance = vout*(mid_pack_voltage-vout)/(acceptable_inductor_ripple_current*fsw_post*mid_pack_voltage)
print(calculated_inductance*10**6) #uH
actual_inductance = 22*10**-6 #H
actual_ripple_current = ((max_pack_voltage/actual_transformer_turns_ratio-vout)*(vout/(max_pack_voltage/actual_transformer_turns_ratio)))/(fsw_post*actual_inductance)
print(actual_ripple_current)

cout = actual_ripple_current/(8*fsw_post*acceptable_dvout)
actual_cout = np.array([[2, 0.1, 0.1],[2, 4.7, 1],[1, 100, 10]])
actual_cout[:, 1] = actual_cout[:, 1]*10**-6 # Convert to F
actual_cout[:, 2] = actual_cout[:, 2]*10**-3 # Convert to Ohms
print(actual_cout)

capacitances = equiv_capacitance(actual_cout, fsw_post)
effective_capacitance = capacitances.equivalent_capacitance
effective_esr = capacitances.equivalent_esr
print(effective_capacitance)
print(effective_esr)

32.14850119152139
10.241187384044528
[[2.0e+00 1.0e-07 1.0e-04]
 [2.0e+00 4.7e-06 1.0e-03]
 [1.0e+00 1.0e-04 1.0e-02]]
0.00010875078669616179
0.008322035710647473


In [11]:
# Ramp Generator
def triangle_wave_calculator(bias_current, vdd, vss, vref, vih, vil):
    ramp_r2 = min(vref/(100*bias_current), (vdd-vref)/(100*bias_current))
    ramp_r1 = ramp_r2*((vih-vil)/(vdd-vss))
    ramp_r3 = vref*(((vih-vref)/(ramp_r1))+((vss-vref)/(ramp_r2)))**-1
    return [ramp_r1, ramp_r2, ramp_r3]

print(triangle_wave_calculator(80*10**-9, 2.46, 0, 1.23, 2.46, 0))





ZeroDivisionError: 0.0 cannot be raised to a negative power

In [ ]:
def gain_at_crossover(v_filter, v_ramp, C, Rc, L, Rl, rout, f_crossover):    
    # https://www.ti.com/lit/an/slva301/slva301.pdf?ts=1745660332504
    # Equation 17
    b1 = C*Rc*2*3.1416*f_crossover
    a1 = Rl/rout
    b2 = 2*3.1416*f_crossover*((L/rout)+Rc*C+Rl*C+((Rl*Rc*C)/rout))
    a2 = -(1 + Rc/rout)*L*C*(2*3.1416*f_crossover)**2

    rational = complex(1, b1)/complex(1 + a1 + a2, b2)
    return 1/((v_filter/v_ramp)*cmath.polar(rational)[0])


In [ ]:
# Compensation Network Calculations
# https://www.ti.com/lit/an/slva301/slva301.pdf?ts=1745660332504

v_ramp = 2.4
vref = 2.5
rbot = 4700 #Ohms
f_crossover = fsw_post/5
inductor_dcr = 0.003 #ohms
rout = vout/iout #TODO check which current to use here and/or sweep

a_mod_ccm = (max_pack_voltage/actual_transformer_turns_ratio)/v_ramp
print(a_mod_ccm)
r1 = rbot*((vout/vref)-1)
print(r1)
f_lc = 1/(2*3.1416*math.sqrt(actual_inductance*effective_capacitance))
print(f_lc)
c3 = 1/(2*3.1416*0.9*f_lc*r1)
print(c3)
r3 = 1/(2*3.1416*f_crossover*c3)
print(r3)
crossover_gain = gain_at_crossover(float(max_pack_voltage)/actual_transformer_turns_ratio, v_ramp, effective_capacitance, effective_esr, actual_inductance, inductor_dcr, rout, f_crossover)
print(crossover_gain)
r2 = parallel_resistance(r1, r3)*crossover_gain
print(r2)
c2 = 1/(1.8*3.1416*f_lc*r2)
print(c2)
c1 = 1/(2*3.1416*10*f_crossover*r2)
print(c1)

18.846153846153847
40420.0
3253.8053733404186
1.3445864347672756e-09
11836.693187137775
0.4669799446106843
4275.461550048134
1.2711653012686178e-08
3.7225120389045485e-10


In [19]:
# Isolated Supply Calculations

vin_15_iso = 24 #V
vout_15_iso = 15 #V

cout2 = 4*switching_fet_qg/(0.1*(vout))
print(cout2)

vref_15_iso = 2.5
r_fb_bot = 10 #kOhms
r_fb_top = r_fb_bot*(1-(vout_15_iso/vin_15_iso))
print(r_fb_top)

6.666666666666665e-08
3.75
